# 4.1 Verdelingen — een hypothese over het onderliggende proces

Een verdeling is een hypothese over het proces dat de data heeft gegenereerd.

"Lognormaal met μ=3,2 en σ=0,8" is een beschrijving die je kunt toetsen aan wat je weet over hoe de getallen tot stand zijn gekomen: een lognormale verdeling stelt dat de waarde een *product* is van vele factoren, wat zorgt voor de lange staart (*long tail*). "Gemiddelde 34, standaarddeviatie 41" beschrijft daarentegen vrijwel niets, omdat bij een lognormale verdeling het gemiddelde niet representatief (typisch) is en de standaarddeviatie een symmetrie suggereert die in werkelijkheid ontbreekt.

Drie redenen om een verdeling te fitten in plaats van enkel een gemiddelde te rapporteren (in oplopende volgorde van waarde):

1. **Om eerlijk samen te vatten.** Twee parameters die de daadwerkelijke *vorm* beschrijven, in plaats van twee getallen die een vorm stilzwijgend veronderstellen.
2. **Om te bepalen wat ongebruikelijk is.** Een datapunt is uitsluitend een uitbijter *relatief ten opzichte van een verdeling*. "Drie standaarddeviaties van het gemiddelde" is een uitspraak over de normale verdeling waarvan je vaak niet wist dat je die had aangenomen.
3. **Om twee situaties met elkaar te vergelijken.** Als dezelfde verdelingsfamilie past vóór en na een gebeurtenis (*event*), vertellen de parameters precies *wát* er is veranderd — het tempo (*rate*), de spreiding (*spread*), de schaal (*scale*) — en niet alleen *dát* er iets is veranderd. Dat is de analytische stap waar deze les naartoe bouwt.

Dit notebook beantwoordt de eerste vraag — *welke vormen bestaan er en welk proces brengt elk voort* — aan de hand van gegenereerde data, zodat er nog niets geschat hoeft te worden. De les past deze families vervolgens in vier stappen toe op echte data:

- [04.2](04.2-long-tail%20jeffrey.ipynb) — één familie op één populatie: de *long tail* van taxiritten in New York.
- [04.3](04.3-penguins%20jeffrey.ipynb) — een fit die faalt omdat de populatie in werkelijkheid uit meerdere subgroepen bestaat.
- [04.4](04.4-before-and-after%20jeffrey.ipynb) — dezelfde familie aan weerszijden van een event, en een nuldistributie om te bepalen hoe zeker je mag zijn.
- [04.5](04.5-fitting-the-residual%20jeffrey.ipynb) — een verdeling gefit op wat een model heeft achtergelaten (het residu).

## Wat goad je hiervoor biedt

Les 1 gaf data-beslissingen een vaste plek in een `Pipeline`, les 2 gaf grafieken een thuisbasis in `BasePlot`. Het fitten van verdelingen krijgt dezelfde aanpak, opgebouwd uit drie componenten:

- **`DistributionRegistry`** bevat de families die goad zal proberen — `norm`, `lognorm`, `exponential`, `gamma`, `weibull`, `poisson`, `nbinom`, en enkele andere — elk voorzien van de metadata die een fit vereist: het scipy-object, of de verdeling discreet is, en hoeveel parameters er zijn. Je kunt zelf een familie registreren die niet standaard wordt meegeleverd (`registry.register_distribution("pareto", stats.pareto, is_discrete=False, num_params=3)`). Elke registry start vanuit de standaardset, zodat een registratie in de ene cel niet ongewild doorlekt naar de volgende.
- **`DistributionFitter(registry, seed=42).fit(data, discrete=...)`** fit elke familie van het juiste type via *maximum likelihood* en levert één resultaat per familie op — een `FitResult` met de geschatte parameters, een `frozen_dist` waar je direct uit kunt plotten of steekproeven uit kunt trekken, de *log-likelihood*, en een Kolmogorov–Smirnov (KS) toets; ofwel een `FailedFit` met de reden van uitval. Mislukte fits zijn gewone waarden, waardoor één slecht presterende familie niet de hele vergelijking opblaast. De parameter `discrete` is *jouw* verantwoordelijkheid: tellingen zijn discreet, tijdsduren zijn continu. Maak je hier een vergissing, dan krijg je geen crash — het algoritme fit simpelweg de verkeerde helft van het register. Met `fit_distribution("lognorm", data)` fit je één specifieke familie als je vooraf al weet welke je wilt. De `seed` is toegevoegd omdat de onderliggende optimizer stochastisch is: gebruik een vaste seed wanneer getallen reproduceerbaar moeten zijn (zoals in een les); laat hem in je eigen werk weg: een fit die verschuift tussen runs vertelt je namelijk iets wezenlijks over de data.
- **`fit_table(results)`** zet deze resultaten om in de gerangschikte tabel die je kunt inleveren: familie, parameters, log-likelihood, KS-toetsingsgrootheid en p-waarde, en op welke criteria de verdeling gewonnen heeft.

Er worden bewust twee winnaars aangeduid. **Log-likelihood** beloont een familie die hoge waarschijnlijkheid toekent aan de daadwerkelijk geobserveerde punten — dit criterium wordt gedomineerd door de bulk van de data. **KS** meet het grootste verschil tussen de theoretische en de empirische cumulatieve verdeling — dit is gevoelig voor het midden en relatief blind voor de uiterste staarten. Meestal zijn ze het eens; wanneer ze oneens zijn, is die discrepantie het meest informatieve signaal in de tabel. Het antwoord vind je dan in een visualisatie, niet in het middelen van twee getallen.

De visualisaties: `PlotFits` (het histogram met de beste fits eroverheen), `QQPlot` (gesorteerde data tegen de kwantielen van een gefitte familie — essentieel voor de staarten, waar families verschillen en histogrammen onleesbaar worden), `ECDFPlot` (twee steekproeven op één as zonder verstorende bin-keuzes), en `DistPlot` (teken een verdelingscurve los of over een histogram). Dit zijn allemaal `BasePlot`s: met `create_figure(n_plots=...)` en `plot_on_axes(...)` combineer je ze op dezelfde consistente manier als in de rest van de cursus.


In [ ]:
import numpy as np
from scipy import stats

from goad_toolkit.visualizer import DistPlot, HistogramPlot, PlotSettings

rng = np.random.default_rng(42)

## 4.1.1 Zes verdelingsfamilies, gegenereerd — vóórdat er echte data aan te pas komt

Elke onderstaande familie is gesampled uit een bekende generator en getekend met haar eigen wiskundige curve over de steekproef heen. Wat je ziet is *uitsluitend* de vorm: geen ruis van fitting, geen twijfel of we de parameters goed hebben geschat. De toelichting onder elke naam beschrijft het genererende mechanisme — hét patroon dat je in je eigen data moet leren herkennen vóórdat je begint met fitten.


In [ ]:
families = [
    ("norm", stats.norm(loc=0, scale=1), "sums of many independent contributions"),
    ("lognorm", stats.lognorm(s=0.8, scale=np.exp(1)), "products of many factors"),
    ("expon", stats.expon(scale=1), "time between events arriving at a steady rate"),
    ("poisson", stats.poisson(mu=4), "counts of events in a fixed window, steady rate"),
    ("weibull", stats.weibull_min(c=1.5, scale=1), "time-to-failure with a changing hazard"),
    ("pareto", stats.pareto(b=2.5), "preferential attachment, rich-get-richer"),
]

gallery = PlotSettings(
    figsize=(13, 7),
    title="Six families, generated",
    subplot_titles=[f"{name}\n{mechanism}" for name, _, mechanism in families],
    max_cols=3,
)
host = HistogramPlot(gallery)
fig, axes = host.create_figure(n_plots=len(families))
for ax, (name, dist, mechanism) in zip(axes, families):
    sample = dist.rvs(2000, random_state=rng)
    # discrete=True gives a count one bar per integer, so the bars and the pmf share a scale
    host.plot_on_axes(HistogramPlot(gallery), ax, data=sample, color="lightgrey",
                      discrete=(name == "poisson"))
    host.plot_on_axes(DistPlot(gallery), ax, distribution=dist, color="crimson")
    ax.set_ylabel("")

Twee van deze families zijn discrete tellingen, geen continue metingen: `poisson` heeft uitsluitend massa op gehele getallen (integers), waardoor de curve door deze punten wordt getrokken — een discrete telling heeft immers geen kansdichtheid (*density*), maar enkel een waarschijnlijkheid per discrete waarde.
En twee families zijn op basis van een standaard histogram nauwelijks uit elkaar te houden: `lognorm` en `pareto` zien er beide simpelweg uit als een verdeling met een "lange staart". Het doorslaggevende onderscheid levert de rang-frequentieplot op log-log-assen op (een machtswet/*power law* vormt daar een rechte lijn, een lognormale verdeling buigt af) — zie het hoofdstuk *Distributions* in goad, §5.7, voor als je dit ooit nodig hebt.


## 4.1.2 De centrale limietstelling — sommen convergeren, niets anders hoeft dat te doen

Sommen van *willekeurige variabelen* met een eindige variantie drijven vanzelf naar de normale verdeling naarmate je meer termen optelt. Dat is de reden dat de normale verdeling overal opduikt waar sprake is van optelling van effecten — maar het zegt helemaal niets over producten, wachttijden of tellingen. Die convergeren naar iets heel anders (lognormaal, gamma, Poisson), juist omdat het géén sommen zijn.


In [ ]:
ns = [1, 2, 5, 30]
clt = PlotSettings(
    figsize=(14, 3.2),  # ty: ignore[invalid-argument-type]
    title="Sums of uniforms converge to normal as n grows",
    subplot_titles=[f"sum of {n} uniform draw{'s' if n > 1 else ''}" for n in ns],
    max_cols=4,
    sharex=True,
    sharey=True,
)
host = HistogramPlot(clt)
fig, axes = host.create_figure(n_plots=len(ns))
for ax, n in zip(axes, ns):
    sums = stats.uniform(loc=-1, scale=2).rvs((5000, n), random_state=rng).sum(axis=1)
    sums = (sums - sums.mean()) / sums.std()
    host.plot_on_axes(HistogramPlot(clt), ax, data=sums, color="steelblue")
    host.plot_on_axes(DistPlot(clt), ax, distribution=stats.norm(0, 1), x_range=(-4, 4))
    ax.set_ylabel("")

In [ ]:
# A PRODUCT of the same uniforms does not converge to normal at all: it converges to
# lognormal, because a sum of logs is a sum, and exp() of a sum is a product.
n = 30
sums = stats.uniform(loc=1, scale=1).rvs((5000, n), random_state=rng).sum(axis=1)
products = stats.uniform(loc=1, scale=1).rvs((5000, n), random_state=rng).prod(axis=1)

product = PlotSettings(
    figsize=(12, 3.2),  # ty: ignore[invalid-argument-type]
    title="The CLT is a statement about sums",
    subplot_titles=["sum → normal", "product → long tail", "log(product) → normal again"],
    max_cols=3,
)
host = HistogramPlot(product)
fig, axes = host.create_figure(n_plots=3)
for ax, values in zip(axes, [sums, products, np.log(products)]):
    host.plot_on_axes(HistogramPlot(product), ax, data=values, color="steelblue")
    ax.set_ylabel("")

Het rechterpaneel toont `log(product)`, niet `product` — dat is precies de wiskundige transformatie waar lognormaal naar vernoemd is. Plot je `products` rechtstreeks, dan is het resultaat scheef en voorzien van een lange staart in plaats van een klokvorm, omdat een product van positieve getallen exact het soort variabele is waar de centrale limietstelling niet rechtstreeks op van toepassing is. Neem je echter de logaritme, dan verandert een product in een *som*, waardoor de stelling weer direct opgaat.


---

**Waar dit nu naartoe gaat.** Elke bovenstaande curve werd getekend op basis van vooraf gekozen parameters. De volgende stap is het omgekeerde: echte getallen, een onbekende familie, en een fitter die moet kiezen welke verdeling het beste past en hoe goed die score is. [04.2-long-tail](04.2-long-tail%20jeffrey.ipynb) past dit toe op taxiritten in New York, waar het onderliggende mechanisme een lognormale verdeling voorspelt en de fitter mag bevestigen of ontkrachten.
